# Task 5 – Exploratory Data Analysis (EDA)
## Titanic Dataset | Elevate Labs Data Analyst Internship

**Objective:** Extract insights using visual and statistical exploration.

**Tools:** Python, Pandas, Matplotlib and Seaborn.

**This notebook covers:** dataset inspection, missing values, duplicates, histograms, boxplots, scatterplots, pairplot, heatmap, trends, relationships and summary findings.

Run the cells from top to bottom. The dataset is downloaded automatically if it is not already in the `data` folder. Internet access is required for the first download.


## 1. Import libraries and set up project folders

In [ ]:
from pathlib import Path
import urllib.request
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

# Find or create a writable project folder.
current_dir = Path.cwd()

if current_dir.name.lower() == "notebooks":
    project_dir = current_dir.parent
elif (current_dir / "data").exists() or (current_dir / "notebooks").exists():
    project_dir = current_dir
else:
    # Safe fallback: create the project in your Windows Documents folder.
    project_dir = Path.home() / "Documents" / "Task5_Titanic_EDA"

project_dir.mkdir(parents=True, exist_ok=True)

data_dir = project_dir / "data"
images_dir = project_dir / "images"
reports_dir = project_dir / "reports"

data_dir.mkdir(parents=True, exist_ok=True)
images_dir.mkdir(parents=True, exist_ok=True)
reports_dir.mkdir(parents=True, exist_ok=True)

print("Project folder:", project_dir)
print("Data folder:", data_dir)
print("Images folder:", images_dir)


## 2. Load the Titanic dataset

In [ ]:
DATA_PATH = data_dir / "titanic.csv"
DATA_URL = "https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv"

if not DATA_PATH.exists():
    print("Downloading Titanic dataset...")
    urllib.request.urlretrieve(DATA_URL, DATA_PATH)

df = pd.read_csv(DATA_PATH)

print("Dataset loaded successfully!")
print("Dataset shape:", df.shape)
display(df.head())


## 3. Understand the dataset: info, describe and value_counts

In [ ]:
print("Dataset information:")
df.info()

print("\nStatistical summary:")
display(df.describe())

print("\nCategorical summary:")
display(df.describe(include="object"))

print("\nSurvival counts (0 = Did not survive, 1 = Survived):")
display(df["Survived"].value_counts())

print("\nPassenger class counts:")
display(df["Pclass"].value_counts().sort_index())

print("\nPassenger counts by sex:")
display(df["Sex"].value_counts())

print("\nEmbarkation port counts:")
display(df["Embarked"].value_counts())


## 4. Check missing values and duplicate records

In [ ]:
missing_count = df.isnull().sum()
missing_percent = (df.isnull().mean() * 100).round(2)

missing_table = pd.DataFrame({
    "Missing Values": missing_count,
    "Missing Percentage": missing_percent
}).sort_values("Missing Values", ascending=False)

display(missing_table)

print("Duplicate rows:", df.duplicated().sum())

plt.figure(figsize=(12, 5))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap="viridis")
plt.title("Missing Values in Titanic Dataset")
plt.xlabel("Columns")
plt.tight_layout()
plt.savefig(images_dir / "01_missing_values_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()


### Observation
Check which columns contain the most missing values. In the commonly used Titanic training dataset, `Cabin` and `Age` have missing observations. Do not assume missing values are random.


## 5. Handle missing values

In [ ]:
# Keep the original dataframe unchanged.
eda = df.copy()

# Fill missing Age values with the median.
eda["Age"] = eda["Age"].fillna(eda["Age"].median())

# Fill missing Embarked values with the most frequent category.
eda["Embarked"] = eda["Embarked"].fillna(eda["Embarked"].mode()[0])

# Cabin has many missing values, so exclude it from this EDA copy.
eda = eda.drop(columns=["Cabin"], errors="ignore")

print("Missing values after preparation:")
display(eda.isnull().sum())

print("Prepared dataset shape:", eda.shape)


## 6. Univariate analysis – Histograms and count plots

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(13, 9))

sns.histplot(data=eda, x="Age", bins=30, kde=True, ax=axes[0, 0])
axes[0, 0].set_title("Age Distribution")

sns.histplot(data=eda, x="Fare", bins=30, kde=True, ax=axes[0, 1])
axes[0, 1].set_title("Fare Distribution")

sns.countplot(data=eda, x="Pclass", ax=axes[1, 0])
axes[1, 0].set_title("Passenger Class Count")

sns.countplot(data=eda, x="Survived", ax=axes[1, 1])
axes[1, 1].set_title("Survival Count (0 = No, 1 = Yes)")

plt.tight_layout()
plt.savefig(images_dir / "02_univariate_analysis.png", dpi=150, bbox_inches="tight")
plt.show()


### Observations
- The age histogram shows the distribution of passenger ages.
- The fare histogram helps identify whether ticket fares are skewed.
- The count plots show the number of passengers in each class and survival category.


## 7. Outlier detection – Boxplots

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

sns.boxplot(data=eda, x="Age", ax=axes[0])
axes[0].set_title("Age Boxplot")

sns.boxplot(data=eda, x="Fare", ax=axes[1])
axes[1].set_title("Fare Boxplot")

plt.tight_layout()
plt.savefig(images_dir / "03_boxplots.png", dpi=150, bbox_inches="tight")
plt.show()


### Observation
Points outside the whiskers are potential outliers. They should be investigated, not automatically deleted. Ticket fares commonly have a long upper tail.


## 8. Bivariate analysis – Relationships with survival

In [ ]:
survival_by_sex = eda.groupby("Sex")["Survived"].mean() * 100
survival_by_class = eda.groupby("Pclass")["Survived"].mean() * 100
survival_by_embarked = eda.groupby("Embarked")["Survived"].mean() * 100

print("Survival rate by sex (%):")
display(survival_by_sex.round(2))

print("Survival rate by passenger class (%):")
display(survival_by_class.round(2))

print("Survival rate by embarkation port (%):")
display(survival_by_embarked.round(2))

fig, axes = plt.subplots(2, 2, figsize=(13, 9))

sns.barplot(data=eda, x="Sex", y="Survived", errorbar=None, ax=axes[0, 0])
axes[0, 0].set_title("Survival Rate by Sex")
axes[0, 0].set_ylabel("Survival Rate")

sns.barplot(data=eda, x="Pclass", y="Survived", errorbar=None, ax=axes[0, 1])
axes[0, 1].set_title("Survival Rate by Passenger Class")
axes[0, 1].set_ylabel("Survival Rate")

sns.boxplot(data=eda, x="Survived", y="Age", ax=axes[1, 0])
axes[1, 0].set_title("Age by Survival Status")

sns.boxplot(data=eda, x="Survived", y="Fare", ax=axes[1, 1])
axes[1, 1].set_title("Fare by Survival Status")

plt.tight_layout()
plt.savefig(images_dir / "04_bivariate_analysis.png", dpi=150, bbox_inches="tight")
plt.show()


### Observations
Compare the survival percentages for male and female passengers, the three passenger classes, and embarkation ports. Use the printed percentages to support your conclusions rather than relying only on the chart.


## 9. Scatterplot – Age versus Fare

In [ ]:
plt.figure(figsize=(9, 6))
sns.scatterplot(data=eda, x="Age", y="Fare", hue="Survived", alpha=0.7)
plt.title("Age vs Fare by Survival Status")
plt.xlabel("Age")
plt.ylabel("Fare")
plt.tight_layout()
plt.savefig(images_dir / "05_age_fare_scatterplot.png", dpi=150, bbox_inches="tight")
plt.show()


### Observation
The scatterplot helps inspect the relationship between age and fare and compare the survival groups. A visible association does not prove causation.


## 10. Multivariate analysis – Pairplot

In [ ]:
numeric_columns = ["Survived", "Pclass", "Age", "SibSp", "Parch", "Fare"]

pairplot = sns.pairplot(
    eda[numeric_columns].dropna(),
    hue="Survived",
    corner=True,
    diag_kind="hist"
)
pairplot.fig.suptitle("Pairplot of Titanic Numerical Features", y=1.02)
pairplot.savefig(images_dir / "06_pairplot.png", dpi=150, bbox_inches="tight")
plt.show()


### Observation
A pairplot displays pairwise relationships between numerical variables. The diagonal shows individual distributions, while the other panels show relationships between pairs of variables.


## 11. Correlation heatmap

In [ ]:
correlation = eda[numeric_columns].corr(numeric_only=True)

plt.figure(figsize=(10, 7))
sns.heatmap(correlation, annot=True, cmap="coolwarm", fmt=".2f", center=0)
plt.title("Correlation Heatmap")
plt.tight_layout()
plt.savefig(images_dir / "07_correlation_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()

display(correlation)


### Observation
The heatmap shows the direction and strength of linear correlations. Correlation values range from -1 to +1. Correlation does not prove causation. Strong correlations among predictor variables may indicate possible multicollinearity; VIF can be used for further investigation.


## 12. Skewness analysis

In [ ]:
skew_values = eda[["Age", "Fare", "SibSp", "Parch"]].skew().sort_values(ascending=False)

print("Skewness values:")
display(skew_values)

for column, value in skew_values.items():
    if value > 0.5:
        print(f"{column}: positively skewed")
    elif value < -0.5:
        print(f"{column}: negatively skewed")
    else:
        print(f"{column}: approximately symmetric")


### Observation
A positive skew means the distribution has a longer right tail. A negative skew means it has a longer left tail. Depending on the analysis objective, skewed variables may be transformed, for example using a log transformation for suitable non-negative data.


## 13. Final summary of findings

In [ ]:
overall_survival_rate = eda["Survived"].mean() * 100
highest_sex = survival_by_sex.idxmax()
highest_sex_rate = survival_by_sex.max()
highest_class = survival_by_class.idxmax()
highest_class_rate = survival_by_class.max()

print("========== TASK 5 EDA SUMMARY ==========")
print(f"Total passengers: {len(eda)}")
print(f"Overall survival rate: {overall_survival_rate:.2f}%")
print(f"Highest survival rate by sex: {highest_sex} ({highest_sex_rate:.2f}%)")
print(f"Highest survival rate by passenger class: Class {highest_class} ({highest_class_rate:.2f}%)")
print(f"Number of duplicate rows: {df.duplicated().sum()}")
print("========================================")


## 14. Conclusion

Exploratory Data Analysis was performed on the Titanic passenger dataset using Python, Pandas, Matplotlib and Seaborn.

The analysis included:
1. Dataset inspection using `.info()`, `.describe()` and `.value_counts()`.
2. Missing-value and duplicate checks.
3. Data preparation for analysis.
4. Histograms and count plots.
5. Boxplots for potential outlier detection.
6. Bivariate analysis of survival by passenger characteristics.
7. Scatterplot, pairplot and correlation heatmap.
8. Skewness and correlation review.

The notebook's calculated values and charts should be used when writing the final report. The data is observational, so relationships do not establish causation.


## 15. Save the cleaned analysis dataset (optional)

In [ ]:
cleaned_path = data_dir / "titanic_eda_cleaned.csv"
eda.to_csv(cleaned_path, index=False)
print("Cleaned analysis dataset saved to:", cleaned_path)
print("Charts saved in:", images_dir)
